# Ejemplo completo: Árbol de Decisión con MLflow

Este notebook toma el ejemplo de clasificación con `bank.csv` y le incorpora MLflow para registrar experimentos, parámetros, métricas, modelo entrenado y artefactos como la imagen del árbol de decisión.

## Objetivo didáctico

Comparar diferentes configuraciones de un árbol de decisión y observar los resultados desde la interfaz de MLflow.

## 2. Importación de librerías

In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import mlflow
import mlflow.sklearn

## 3. Configuración inicial de MLflow

El experimento se guardará localmente en la carpeta `mlruns/`. Luego podrá visualizarlo ejecutando `mlflow ui` desde la terminal.

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name="arbol_decision_bank")

2026/05/14 16:19:13 INFO mlflow.tracking.fluent: Experiment with name 'arbol_decision_bank' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1778793553422, experiment_id='2', last_update_time=1778793553422, lifecycle_stage='active', name='arbol_decision_bank', tags={}, trace_location=None, workspace='default'>

## 4. Carga del dataset


In [15]:
# --- 1. Carga y Preparación del Dataset bank.csv ---
df = pd.read_csv("data/bank.csv", delimiter=";")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


## 5. Revisión general del dataset

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


In [17]:
print("Filas y columnas:", df.shape)
df.head()

Filas y columnas: (4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


## 6. Preparación de la variable objetivo

La variable `y` indica si el cliente suscribió o no el depósito. Se transforma:

- `yes` → `1`
- `no` → `0`

In [18]:
df["y"] = df["y"].map({"yes": 1, "no": 0})

print("--- Distribución de la Variable Objetivo 'y' después del mapeo ---")
print(df["y"].value_counts())

--- Distribución de la Variable Objetivo 'y' después del mapeo ---
y
0    4000
1     521
Name: count, dtype: int64


## 7. Codificación de variables categóricas

Se utiliza `pd.get_dummies()` para convertir variables categóricas en variables numéricas.

In [19]:
variables_categorical = [
    "job", "marital", "education", "default", "housing",
    "loan", "contact", "month", "poutcome"
]

df_encoded = pd.get_dummies(
    df,
    columns=variables_categorical,
    drop_first=True
)

df_encoded.head()

,age,balance,day,duration,campaign,pdays,previous,y,job_blue-collar,job_entrepreneur,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,30,1787,19,79,1,-1,0,0,False,False,...,False,False,False,False,False,True,False,False,False,True
1,33,4789,11,220,1,339,4,0,False,False,...,False,False,False,True,False,False,False,False,False,False
2,35,1350,16,185,1,330,1,0,False,False,...,False,False,False,False,False,False,False,False,False,False
3,30,1476,3,199,4,-1,0,0,False,False,...,False,True,False,False,False,False,False,False,False,True
4,59,0,5,226,1,-1,0,0,True,False,...,False,False,False,True,False,False,False,False,False,True


## 8. Definición de variables predictoras y variable objetivo

Se excluye `duration` porque en muchos análisis de este dataset se considera una variable problemática para predicción previa, ya que se conoce después de realizada la llamada.

In [20]:
X = df_encoded.drop(columns=["y", "duration"])
y = df_encoded["y"]

print("Variables predictoras:", X.shape[1])
print("Total de registros:", X.shape[0])

Variables predictoras: 41
Total de registros: 4521


In [22]:
df_encoded.dtypes

age                    int64
balance                int64
day                    int64
duration               int64
campaign               int64
pdays                  int64
previous               int64
y                      int64
job_blue-collar         bool
job_entrepreneur        bool
job_housemaid           bool
job_management          bool
job_retired             bool
job_self-employed       bool
job_services            bool
job_student             bool
job_technician          bool
job_unemployed          bool
job_unknown             bool
marital_married         bool
marital_single          bool
education_secondary     bool
education_tertiary      bool
education_unknown       bool
default_yes             bool
housing_yes             bool
loan_yes                bool
contact_telephone       bool
contact_unknown         bool
month_aug               bool
month_dec               bool
month_feb               bool
month_jan               bool
month_jul               bool
month_jun     

## 9. División de datos en entrenamiento y prueba

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("--- Tamaños de los Conjuntos de Datos ---")
print(f"Tamaño del conjunto de entrenamiento: {len(X_train)} muestras")
print(f"Tamaño del conjunto de prueba: {len(X_test)} muestras")

--- Tamaños de los Conjuntos de Datos ---
Tamaño del conjunto de entrenamiento: 3616 muestras
Tamaño del conjunto de prueba: 905 muestras


## 10. Experimento individual con MLflow

Esta celda entrena un árbol de decisión y registra en MLflow:

- Parámetros del modelo
- Métricas de evaluación
- Modelo entrenado
- Reporte de clasificación

In [11]:
# Parámetros del árbol de decisión
max_depth = 5
min_samples_leaf = 50
min_samples_split = 100
criterion = "gini"
random_state = 42

with mlflow.start_run(run_name="arbol_decision_base"):

    decision_tree_model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        criterion=criterion,
        random_state=random_state
    )

    print("--- Entrenando el Modelo de Árbol de Decisión ---")
    decision_tree_model.fit(X_train, y_train)
    print("Entrenamiento completado.")

    y_pred = decision_tree_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Registrar parámetros en MLflow
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("criterion", criterion)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("variables_predictoras", X.shape[1])

    # Registrar métricas en MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Guardar el modelo entrenado
    mlflow.sklearn.log_model(
        sk_model=decision_tree_model,
        artifact_path="modelo_arbol_decision"
    )

    # Crear y guardar reporte de clasificación como artefacto
    reporte = classification_report(
        y_test,
        y_pred,
        target_names=["No Suscribe (0)", "Suscribe (1)"]
    )

    with open("reporte_clasificacion.txt", "w", encoding="utf-8") as archivo:
        archivo.write(reporte)

    mlflow.log_artifact("reporte_clasificacion.txt")

    print("--- Resultados de la Evaluación del Modelo ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("Reporte de Clasificación:")
    print(reporte)    

--- Entrenando el Modelo de Árbol de Decisión ---
Entrenamiento completado.


2026/05/14 16:19:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:19:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- Resultados de la Evaluación del Modelo ---
Accuracy: 0.8917
Precision: 0.6000
Recall: 0.1731
F1-score: 0.2687
Reporte de Clasificación:
                 precision    recall  f1-score   support

No Suscribe (0)       0.90      0.99      0.94       801
   Suscribe (1)       0.60      0.17      0.27       104

       accuracy                           0.89       905
      macro avg       0.75      0.58      0.61       905
   weighted avg       0.87      0.89      0.86       905

🏃 View run arbol_decision_base at: http://127.0.0.1:5000/#/experiments/2/runs/c0974429bdf1430096ba54476b66aa02
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


## 11. Comparación de varios experimentos

Esta parte es útil para clase. Los estudiantes pueden cambiar los hiperparámetros y observar cuál configuración produce mejores resultados.

In [ ]:
experimentos = [
    {"max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 50, "criterion": "gini"},
    {"max_depth": 5, "min_samples_leaf": 50, "min_samples_split": 100, "criterion": "gini"},
    {"max_depth": 7, "min_samples_leaf": 30, "min_samples_split": 80, "criterion": "entropy"},
    {"max_depth": 10, "min_samples_leaf": 10, "min_samples_split": 40, "criterion": "gini"},
]

